# Can we predict `failed_90d` after an inspection?
Plan: inspect data, check for leakage (esp. `parts_replaced`), validate chronologically, compare a baseline against simple models, interpret.

In [ ]:
import os; os.environ.setdefault('MPLCONFIGDIR', '/tmp/mpl')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.inspection import permutation_importance

d = pd.read_csv('data.csv', parse_dates=['inspected_at']).sort_values('inspected_at').reset_index(drop=True)
print(d.shape, '| unique assets:', d.asset_tag.nunique(), '| missing cells:', int(d.isna().sum().sum()))
print('failure rate: %.3f' % d.failed_90d.mean())
d.describe().T

## Leakage check: `parts_replaced`
Per the data dictionary it is the number of parts replaced on the work order *that closed this inspection record*. That work order can be closed after a failure and repair, so it may encode the outcome instead of predicting it.

In [ ]:
d.groupby('parts_replaced').failed_90d.agg(['mean', 'size'])

Every record with 2 or more parts replaced failed, and none with 0 did. That near-deterministic pattern is typical of a post-outcome field, so I **exclude `parts_replaced`** from the real models. It is not known at inspection time. I include it once below, labelled LEAKY, to show how much it inflates the score.

## Other EDA
Each asset appears once, so asset-level grouping isn't needed.

In [ ]:
num = ['vibration_rms', 'bearing_temp_c', 'ambient_temp_c', 'meter_hours', 'load_pct']
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
d.groupby(d.inspected_at.dt.to_period('Q')).failed_90d.mean().plot.bar(ax=ax[0], title='Failure rate by inspection quarter')
d.groupby('site').failed_90d.mean().plot.bar(ax=ax[1], title='Failure rate by site')
plt.tight_layout(); plt.show()
d.groupby('failed_90d')[num].mean().T

The failure rate moves a lot over time, so a random split would be optimistic and the base rate drifts. I use a **chronological split**: train on the first 70%, test on the latest 30%. It is also worth confirming with the data owner that the most recent inspections have a complete 90-day follow-up window.

In [ ]:
cut = int(len(d) * .7); tr, te = d.iloc[:cut], d.iloc[cut:]
print('train rate %.3f | test rate %.3f | test starts %s' % (tr.failed_90d.mean(), te.failed_90d.mean(), te.inspected_at.min().date()))

def fit_eval(name, model, cols):
    cat = [c for c in cols if c == 'site']
    pre = ColumnTransformer([('n', StandardScaler(), [c for c in cols if c not in cat])] +
                            ([('c', OneHotEncoder(handle_unknown='ignore'), cat)] if cat else []))
    m = make_pipeline(pre, model).fit(tr[cols], tr.failed_90d)
    p = m.predict_proba(te[cols])[:, 1]
    return dict(model=name, roc_auc=roc_auc_score(te.failed_90d, p), pr_auc=average_precision_score(te.failed_90d, p)), m

hgb = lambda: HistGradientBoostingClassifier(max_depth=3, learning_rate=.05, max_iter=200, random_state=0)
specs = [('baseline (prior)', DummyClassifier(), num),
         ('logistic', LogisticRegression(max_iter=1000), num),
         ('logistic + site', LogisticRegression(max_iter=1000), num + ['site']),
         ('gradient boosting', hgb(), num),
         ('LEAKY: logistic incl. parts_replaced', LogisticRegression(max_iter=1000), num + ['parts_replaced'])]
rows, fits = [], {}
for name, mod, cols in specs:
    r, m = fit_eval(name, mod, cols); rows.append(r); fits[name] = m
print('PR-AUC of a random scorer = test base rate = %.3f' % te.failed_90d.mean())
pd.DataFrame(rows).set_index('model').round(3)

## Which features matter?

In [ ]:
pi = permutation_importance(fits['gradient boosting'], te[num], te.failed_90d, scoring='roc_auc', n_repeats=20, random_state=0)
pd.Series(pi.importances_mean, num).sort_values().plot.barh(title='Permutation importance (drop in test ROC-AUC)')
plt.show()
pd.Series(fits['logistic'][-1].coef_[0], num, name='standardised coef').round(3)

## Bootstrap uncertainty on test ROC-AUC (logistic)

In [ ]:
p = fits['logistic'].predict_proba(te[num])[:, 1]; y = te.failed_90d.values; rng = np.random.default_rng(0)
b = [roc_auc_score(y[i], p[i]) for i in (rng.integers(0, len(y), len(y)) for _ in range(500))]
print('ROC-AUC 95%% CI: %.3f to %.3f' % tuple(np.percentile(b, [2.5, 97.5])))

## Conclusions
**Short answer: modestly, yes. Test ROC-AUC is about 0.68 for logistic regression and 0.76 for gradient boosting, against 0.50 for the baseline. The signal is weak, and it is far from the near-perfect score you would get by using `parts_replaced`.**

- **Leakage:** `parts_replaced` gives ROC-AUC 0.997 and PR-AUC 0.97 because it is almost certainly recorded after the failure. It is excluded from the models. Confirm with the work-order owner when that field is populated. If it is filled in only after an outcome, it cannot be used for prediction.
- **Real signal:** vibration RMS is the strongest driver, followed by load and bearing temperature. Site adds nothing.
- **Precision is low:** the test PR-AUC is 0.10 to 0.15 against a 5.6% base rate. A model like this can help rank inspections for triage, but it is not reliable enough for automatic decisions.
- **Caveats:**
  - The failure rate drifts a lot over time: train 14.5%, test 5.6%. Check that the late-2025 records have a full 90-day follow-up window, otherwise their labels are biased low.
  - The bootstrap 95% interval for the logistic ROC-AUC is 0.63 to 0.73, and the test set has only about 1,800 rows with roughly 100 failures. The gap to gradient boosting is suggestive but not firmly established, and I have not tuned either model.
  - Plots were not rendered in the saved outputs (see the note in the summary), so re-run the notebook to see them.
